**Create Silver schema**

In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS workspace.03_silver_transformations;

SHOW SCHEMAS IN workspace;

databaseName
03_silver_transformations
bronze_layer
default
information_schema
silver_layer


**Select your Silver schema**

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.`03_silver_transformations`")

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA `03_silver_transformations`")

spark.sql("""
SELECT current_catalog() AS catalog,
       current_schema() AS schema
""").show()

+---------+--------------------+
|  catalog|              schema|
+---------+--------------------+
|workspace|`03_silver_transf...|
+---------+--------------------+



**Confirm Bronze tables**

In [0]:
spark.sql("SHOW TABLES IN workspace.default").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
| default|     bronze_chargers|      false|
| default|  bronze_maintenance|      false|
| default|     bronze_sessions|      false|
| default|     bronze_stations|      false|
| default|shiptrack_week03_...|      false|
| default|shiptrack_week03_...|      false|
+--------+--------------------+-----------+



**Record Bronze counts**

In [0]:
bronze_counts = {
    "stations": spark.table("workspace.default.bronze_stations").count(),
    "maintenance": spark.table("workspace.default.bronze_maintenance").count(),
    "chargers": spark.table("workspace.default.bronze_chargers").count(),
    "sessions": spark.table("workspace.default.bronze_sessions").count()
}

print("Bronze counts:")
for entity, count in bronze_counts.items():
    print(f"{entity}: {count}")

Bronze counts:
stations: 180
maintenance: 18000
chargers: 1200
sessions: 300000


**Inspect Stations schema**

In [0]:
spark.table("workspace.default.bronze_stations").printSchema()

root
 |-- physical_record_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- city_band: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- site_type: string (nullable = true)
 |-- operator_code: string (nullable = true)
 |-- connector_capacity: integer (nullable = true)
 |-- operating_start_hour: integer (nullable = true)
 |-- operating_end_hour: integer (nullable = true)
 |-- is_24x7: boolean (nullable = true)
 |-- commission_date: date (nullable = true)
 |-- station_status: string (nullable = true)
 |-- latitude_band: double (nullable = true)
 |-- longitude_band: double (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)



**Inspect Maintenance schema**

In [0]:
spark.table("workspace.default.bronze_maintenance").printSchema()

root
 |-- physical_record_id: string (nullable = true)
 |-- maintenance_id: string (nullable = true)
 |-- incident_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- fault_category: string (nullable = true)
 |-- fault_code: string (nullable = true)
 |-- severity: string (nullable = true)
 |-- status_after: string (nullable = true)
 |-- related_fault_event_id: string (nullable = true)
 |-- planned_flag: boolean (nullable = true)
 |-- notes_code: string (nullable = true)
 |-- batch_date: date (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)



**Inspect Chargers schema**

In [0]:
spark.table("workspace.default.bronze_chargers").printSchema()

root
 |-- charger_id: string (nullable = true)
 |-- charger_label: string (nullable = true)
 |-- connector_position: long (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- firmware_major: long (nullable = true)
 |-- install_date: string (nullable = true)
 |-- manufacturer_band: string (nullable = true)
 |-- operational_status: string (nullable = true)
 |-- physical_record_id: string (nullable = true)
 |-- rated_power_kw: double (nullable = true)
 |-- smart_meter_enabled: boolean (nullable = true)
 |-- source_system: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- run_id: string (nullable = true)



**Inspect Sessions schema**

In [0]:
spark.table("workspace.default.bronze_sessions").printSchema()

root
 |-- physical_record_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- charger_id: string (nullable = true)
 |-- vehicle_class: string (nullable = true)
 |-- connector_type: string (nullable = true)
 |-- arrival_ts: timestamp (nullable = true)
 |-- charge_start_ts: timestamp (nullable = true)
 |-- charge_end_ts: timestamp (nullable = true)
 |-- departure_ts: timestamp (nullable = true)
 |-- final_status: string (nullable = true)
 |-- end_reason: string (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- start_soc_pct: long (nullable = true)
 |-- end_soc_pct: long (nullable = true)
 |-- tariff_band: string (nullable = true)
 |-- tariff_rate_inr_per_kwh: double (nullable = true)
 |-- meter_quality_flag: string (nullable = true)
 |-- batch_date: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

**Create Silver Stations**

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.`03_silver_transformations`.silver_stations_candidate
USING DELTA
AS
SELECT
    *,
    current_timestamp() AS _candidate_created_at,
    'ev_silver_candidate_v1.0' AS _candidate_schema_version
FROM workspace.default.bronze_stations
""")

print("silver_stations_candidate created successfully")

silver_stations_candidate created successfully


**Inspect Silver Stations**

In [0]:
spark.sql("""
SELECT *
FROM workspace.`03_silver_transformations`.silver_stations_candidate
LIMIT 10
""").show(truncate=False)

+------------------+----------+--------------------------------------+---------+----------------+-------------------+-------------+------------------+--------------------+------------------+-------+---------------+--------------+-------------+--------------+-----------------------+--------------------------+----------------------------------------------------+-------+--------------------------+-------------------------+
|physical_record_id|station_id|station_name                          |city_band|zone            |site_type          |operator_code|connector_capacity|operating_start_hour|operating_end_hour|is_24x7|commission_date|station_status|latitude_band|longitude_band|source_system          |ingestion_time            |source_file                                         |run_id |_candidate_created_at     |_candidate_schema_version|
+------------------+----------+--------------------------------------+---------+----------------+-------------------+-------------+------------------+--

**Create Silver Maintenance**

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.`03_silver_transformations`.silver_maintenance_candidate
USING DELTA
AS
SELECT
    *,
    current_timestamp() AS _candidate_created_at,
    'ev_silver_candidate_v1.0' AS _candidate_schema_version
FROM workspace.default.bronze_maintenance
""")

print("silver_maintenance_candidate created successfully")

silver_maintenance_candidate created successfully


**Inspect Silver Maintenance**

In [0]:
spark.sql("""
SELECT *
FROM workspace.`03_silver_transformations`.silver_maintenance_candidate
LIMIT 10
""").show(truncate=False)

+------------------+--------------+-----------+----------+----------+-------------------+------------------+--------------+----------+--------+------------+----------------------+------------+------------------------------------+----------+--------------------+--------------------------+-------------------------------------------------------+-------+--------------------------+-------------------------+
|physical_record_id|maintenance_id|incident_id|station_id|charger_id|event_ts           |event_type        |fault_category|fault_code|severity|status_after|related_fault_event_id|planned_flag|notes_code                          |batch_date|source_system       |ingestion_time            |source_file                                            |run_id |_candidate_created_at     |_candidate_schema_version|
+------------------+--------------+-----------+----------+----------+-------------------+------------------+--------------+----------+--------+------------+----------------------+---------

**Create Silver Chargers**

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.`03_silver_transformations`.silver_chargers_candidate
USING DELTA
AS
SELECT
    *,
    current_timestamp() AS _candidate_created_at,
    'ev_silver_candidate_v1.0' AS _candidate_schema_version
FROM workspace.default.bronze_chargers
""")

print("silver_chargers_candidate created successfully")

silver_chargers_candidate created successfully


**Inspect Silver Chargers**

In [0]:
spark.sql("""
SELECT *
FROM workspace.`03_silver_transformations`.silver_chargers_candidate
LIMIT 10
""").show(truncate=False)

+----------+-------------+------------------+--------------+--------------+------------+-----------------+------------------+------------------+--------------+-------------------+-----------------------+----------+-------------------------+-----------------------------------------------------+-------+-------------------------+-------------------------+
|charger_id|charger_label|connector_position|connector_type|firmware_major|install_date|manufacturer_band|operational_status|physical_record_id|rated_power_kw|smart_meter_enabled|source_system          |station_id|ingestion_time           |source_file                                          |run_id |_candidate_created_at    |_candidate_schema_version|
+----------+-------------+------------------+--------------+--------------+------------+-----------------+------------------+------------------+--------------+-------------------+-----------------------+----------+-------------------------+--------------------------------------------------

**Create Silver Sessions**

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.`03_silver_transformations`.silver_sessions_candidate
USING DELTA
AS
SELECT
    *,
    current_timestamp() AS _candidate_created_at,
    'ev_silver_candidate_v1.0' AS _candidate_schema_version
FROM workspace.default.bronze_sessions
""")

print("silver_sessions_candidate created successfully")

silver_sessions_candidate created successfully


**Inspect Silver Sessions**

In [0]:
spark.sql("""
SELECT *
FROM workspace.`03_silver_transformations`.silver_sessions_candidate
LIMIT 10
""").show(truncate=False)

+------------------+------------+----------+----------+-------------+--------------+-----------------------+-----------------------+-----------------------+-----------------------+------------+--------------+----------+-------------+-----------+-----------+-----------------------+------------------+----------+-------------------------+--------------------------+----------------+-------+--------------------------+-------------------------+
|physical_record_id|session_id  |station_id|charger_id|vehicle_class|connector_type|arrival_ts             |charge_start_ts        |charge_end_ts          |departure_ts           |final_status|end_reason    |energy_kwh|start_soc_pct|end_soc_pct|tariff_band|tariff_rate_inr_per_kwh|meter_quality_flag|batch_date|source_system            |ingestion_time            |source_file     |run_id |_candidate_created_at     |_candidate_schema_version|
+------------------+------------+----------+----------+-------------+--------------+-----------------------+------

**Confirm all 4 Silver tables**

In [0]:
spark.sql("""
SHOW TABLES IN workspace.`03_silver_transformations`
""").show()

+--------------------+--------------------+-----------+
|            database|           tableName|isTemporary|
+--------------------+--------------------+-----------+
|`03_silver_transf...|silver_chargers_c...|      false|
|`03_silver_transf...|silver_maintenanc...|      false|
|`03_silver_transf...|silver_sessions_c...|      false|
|`03_silver_transf...|silver_stations_c...|      false|
+--------------------+--------------------+-----------+



**Check all 4 Silver counts**

In [0]:
silver_counts = {
    "stations": spark.table(
        "workspace.`03_silver_transformations`.silver_stations_candidate"
    ).count(),

    "maintenance": spark.table(
        "workspace.`03_silver_transformations`.silver_maintenance_candidate"
    ).count(),

    "chargers": spark.table(
        "workspace.`03_silver_transformations`.silver_chargers_candidate"
    ).count(),

    "sessions": spark.table(
        "workspace.`03_silver_transformations`.silver_sessions_candidate"
    ).count()
}

print("Silver Candidate counts:")
for entity, count in silver_counts.items():
    print(f"{entity}: {count}")

Silver Candidate counts:
stations: 180
maintenance: 18000
chargers: 1200
sessions: 300000


**Complete reconciliation**

In [0]:
reconciliation = spark.sql("""
WITH counts AS (

    SELECT
        'stations' AS entity,
        (SELECT COUNT(*)
         FROM workspace.default.bronze_stations) AS bronze_rows,
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_stations_candidate) AS candidate_rows

    UNION ALL

    SELECT
        'maintenance',
        (SELECT COUNT(*)
         FROM workspace.default.bronze_maintenance),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_maintenance_candidate)

    UNION ALL

    SELECT
        'chargers',
        (SELECT COUNT(*)
         FROM workspace.default.bronze_chargers),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_chargers_candidate)

    UNION ALL

    SELECT
        'sessions',
        (SELECT COUNT(*)
         FROM workspace.default.bronze_sessions),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_sessions_candidate)
)

SELECT
    entity,
    bronze_rows,
    candidate_rows,
    candidate_rows - bronze_rows AS difference,
    CASE
        WHEN bronze_rows = candidate_rows THEN 'PASS'
        ELSE 'CHECK'
    END AS status
FROM counts
ORDER BY entity
""")

reconciliation.show()

+-----------+-----------+--------------+----------+------+
|     entity|bronze_rows|candidate_rows|difference|status|
+-----------+-----------+--------------+----------+------+
|   chargers|       1200|          1200|         0|  PASS|
|maintenance|      18000|         18000|         0|  PASS|
|   sessions|     300000|        300000|         0|  PASS|
|   stations|        180|           180|         0|  PASS|
+-----------+-----------+--------------+----------+------+



**Check Delta format**

In [0]:
for table in [
    "silver_stations_candidate",
    "silver_maintenance_candidate",
    "silver_chargers_candidate",
    "silver_sessions_candidate"
]:
    print(f"\n--- {table} ---")
    spark.sql(
        f"DESCRIBE DETAIL workspace.`03_silver_transformations`.{table}"
    ).select("format", "numFiles", "sizeInBytes").show()


--- silver_stations_candidate ---
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
| delta|       1|      10940|
+------+--------+-----------+


--- silver_maintenance_candidate ---
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
| delta|       1|     200942|
+------+--------+-----------+


--- silver_chargers_candidate ---
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
| delta|       1|      14722|
+------+--------+-----------+


--- silver_sessions_candidate ---
+------+--------+-----------+
|format|numFiles|sizeInBytes|
+------+--------+-----------+
| delta|       1|    8023416|
+------+--------+-----------+



**Check lineage columns**

In [0]:
spark.sql("""
SELECT
    source_file,
    run_id,
    ingestion_time
FROM workspace.`03_silver_transformations`.silver_stations_candidate
LIMIT 10
""").show(truncate=False)

+----------------------------------------------------+-------+--------------------------+
|source_file                                         |run_id |ingestion_time            |
+----------------------------------------------------+-------+--------------------------+
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Vol

**Check lineage for all tables**

In [0]:
tables = [
    "silver_stations_candidate",
    "silver_maintenance_candidate",
    "silver_chargers_candidate",
    "silver_sessions_candidate"
]

for table in tables:
    print(f"\n===== {table} =====")
    
    spark.sql(f"""
        SELECT
            source_file,
            run_id,
            ingestion_time
        FROM workspace.`03_silver_transformations`.{table}
        LIMIT 3
    """).show(truncate=False)


===== silver_stations_candidate =====
+----------------------------------------------------+-------+--------------------------+
|source_file                                         |run_id |ingestion_time            |
+----------------------------------------------------+-------+--------------------------+
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
|dbfs:/Volumes/ev-charge/default/ev-data/stations.csv|run_001|2026-08-01 09:25:58.346709|
+----------------------------------------------------+-------+--------------------------+


===== silver_maintenance_candidate =====
+-------------------------------------------------------+-------+--------------------------+
|source_file                                            |run_id |ingestion_time            |
+-------------------------------------------------------+-------+--------------------------+
|dbfs:/Vo

**Check for missing source files**

In [0]:
for table in tables:
    print(f"\n===== {table} =====")
    
    spark.sql(f"""
        SELECT COUNT(*) AS missing_source_file
        FROM workspace.`03_silver_transformations`.{table}
        WHERE source_file IS NULL
    """).show()


===== silver_stations_candidate =====
+-------------------+
|missing_source_file|
+-------------------+
|                  0|
+-------------------+


===== silver_maintenance_candidate =====
+-------------------+
|missing_source_file|
+-------------------+
|                  0|
+-------------------+


===== silver_chargers_candidate =====
+-------------------+
|missing_source_file|
+-------------------+
|                  0|
+-------------------+


===== silver_sessions_candidate =====
+-------------------+
|missing_source_file|
+-------------------+
|                  0|
+-------------------+



**Check run IDs**

In [0]:
for table in tables:
    print(f"\n===== {table} =====")
    
    spark.sql(f"""
        SELECT run_id, COUNT(*) AS row_count
        FROM workspace.`03_silver_transformations`.{table}
        GROUP BY run_id
        ORDER BY run_id
    """).show()


===== silver_stations_candidate =====
+-------+---------+
| run_id|row_count|
+-------+---------+
|run_001|      180|
+-------+---------+


===== silver_maintenance_candidate =====
+-------+---------+
| run_id|row_count|
+-------+---------+
|run_001|    18000|
+-------+---------+


===== silver_chargers_candidate =====
+-------+---------+
| run_id|row_count|
+-------+---------+
|run_001|     1200|
+-------+---------+


===== silver_sessions_candidate =====
+-------+---------+
| run_id|row_count|
+-------+---------+
|run_001|   300000|
+-------+---------+



**Controlled rerun**

In [0]:
# rerun cells 18,22,26,30 and then 38 there it shouls remain same count
# stations 180
# maintenance  18000
# chargers   1200
# sessions  300000

**Final Week-5 status**

In [0]:
final_check = spark.sql("""
WITH counts AS (

    SELECT
        'stations' AS entity,
        (SELECT COUNT(*) FROM workspace.default.bronze_stations) AS bronze_rows,
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_stations_candidate) AS candidate_rows

    UNION ALL

    SELECT
        'maintenance',
        (SELECT COUNT(*) FROM workspace.default.bronze_maintenance),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_maintenance_candidate)

    UNION ALL

    SELECT
        'chargers',
        (SELECT COUNT(*) FROM workspace.default.bronze_chargers),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_chargers_candidate)

    UNION ALL

    SELECT
        'sessions',
        (SELECT COUNT(*) FROM workspace.default.bronze_sessions),
        (SELECT COUNT(*)
         FROM workspace.`03_silver_transformations`.silver_sessions_candidate)
)

SELECT
    entity,
    bronze_rows,
    candidate_rows,
    CASE
        WHEN bronze_rows = candidate_rows THEN 'PASS'
        ELSE 'FAIL'
    END AS reconciliation_status
FROM counts
ORDER BY entity
""")

final_check.show()

+-----------+-----------+--------------+---------------------+
|     entity|bronze_rows|candidate_rows|reconciliation_status|
+-----------+-----------+--------------+---------------------+
|   chargers|       1200|          1200|                 PASS|
|maintenance|      18000|         18000|                 PASS|
|   sessions|     300000|        300000|                 PASS|
|   stations|        180|           180|                 PASS|
+-----------+-----------+--------------+---------------------+

